# Part 3: Data Analytics — Labor Stats Quest

This notebook loads the BLS time-series data (Part 1) and US population data (Part 2)
from S3 and generates three analytical reports.

**Data Sources:**
- BLS data: `s3://<bucket>/bls/pr/pr.data.0.Current`
- Population data: `s3://<bucket>/population/us_population.json`

In [ ]:
# Install dependencies if running locally (not needed in Lambda)
# !pip install boto3 pandas requests

In [ ]:
import io
import json
import os

import boto3
import pandas as pd

S3_BUCKET = os.environ.get('S3_BUCKET_NAME', 'labor-stats-data')
BLS_KEY = 'bls/pr/pr.data.0.Current'
POP_KEY = 'population/us_population.json'

s3 = boto3.client('s3')
print(f'Using bucket: {S3_BUCKET}')

## Load Data

In [ ]:
# Load BLS time-series data
obj = s3.get_object(Bucket=S3_BUCKET, Key=BLS_KEY)
content = obj['Body'].read().decode('utf-8')

bls_df = pd.read_csv(io.StringIO(content), sep='\t', dtype=str)

# IMPORTANT: Strip whitespace — BLS data has trailing spaces in column names & values
bls_df.columns = [c.strip() for c in bls_df.columns]
for col in bls_df.columns:
    bls_df[col] = bls_df[col].str.strip()

bls_df['year'] = bls_df['year'].astype(int)
bls_df['value'] = pd.to_numeric(bls_df['value'], errors='coerce')

print(f'BLS records: {len(bls_df)}')
bls_df.head()

In [ ]:
# Load population data
obj = s3.get_object(Bucket=S3_BUCKET, Key=POP_KEY)
payload = json.loads(obj['Body'].read().decode('utf-8'))

records = payload.get('data', payload)
pop_df = pd.DataFrame(records)
pop_df.columns = [c.strip() for c in pop_df.columns]

# Rename API fields to lowercase
pop_df.rename(columns={'Year': 'year', 'Population': 'population'}, inplace=True)
pop_df['year'] = pop_df['year'].astype(int)
pop_df['population'] = pd.to_numeric(pop_df['population'], errors='coerce')

print(f'Population records: {len(pop_df)}')
pop_df.sort_values('year').head()

## Report 1: Population Mean & Std-Dev (2013–2018)

In [ ]:
filtered = pop_df[(pop_df['year'] >= 2013) & (pop_df['year'] <= 2018)]

mean_pop = filtered['population'].mean()
std_pop  = filtered['population'].std()

print('=== Report 1: US Population Statistics (2013-2018 inclusive) ===')
print(f'Records: {len(filtered)}')
print(f'Mean Population : {mean_pop:,.2f}')
print(f'Std Dev Population: {std_pop:,.2f}')

filtered[['year','population']].sort_values('year')

## Report 2: Best Year per Series ID

In [ ]:
quarterly = bls_df[bls_df['period'].str.startswith('Q')]

grouped = (
    quarterly
    .groupby(['series_id', 'year'])['value']
    .sum()
    .reset_index()
)

idx = grouped.groupby('series_id')['value'].idxmax()
best_year_df = grouped.loc[idx].reset_index(drop=True)
best_year_df.rename(columns={'value': 'summed_value'}, inplace=True)

print(f'=== Report 2: Best Year per Series ID ({len(best_year_df)} series) ===')
best_year_df.head(20)

## Report 3: PRS30006032 Q01 Values with Population

In [ ]:
# Filter BLS for the target series + period
target = bls_df[
    (bls_df['series_id'] == 'PRS30006032') &
    (bls_df['period'] == 'Q01')
][['series_id','year','period','value']].copy()

# Left-join with population on year
report3 = target.merge(pop_df[['year','population']], on='year', how='left')
report3 = report3.sort_values('year').reset_index(drop=True)

print('=== Report 3: PRS30006032 | Q01 | with Population (where available) ===')
report3

---
## Summary
| Report | Key Finding |
|--------|-------------|
| 1 | Mean and Std-Dev of US population 2013-2018 computed above |
| 2 | Best year (highest annual summed value) identified per series |
| 3 | PRS30006032 Q01 values joined with population data |